# 02 — Agent Evaluation

**PPE Compliance Agent — ITAI 1378 Final Project**

This notebook evaluates the agent at two levels, per course requirements:

1. **Component level** — the CV model's own metrics (mAP, precision, recall) from training
2. **System level** — task success rate across 10+ test scenarios, robustness to bad inputs, efficiency, and honest failure analysis

Run this from the repo root (or adjust `sys.path` if running elsewhere).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root, so `agents` and `tools` import cleanly

from agents.ppe_compliance_agent import PPEComplianceAgent

## 1. Component-Level Metrics (from training)

In [ ]:
component_metrics = {
    "mAP@0.5": 0.900,
    "mAP@0.5:0.95": 0.608,
    "precision": 0.934,
    "recall": 0.825,
    "epochs_trained": 75,
    "training_time_min": 76.4,
}
for k, v in component_metrics.items():
    print(f"{k:20s}: {v}")

## 2. System-Level Evaluation — Task Success Rate

Runs the full agent (all 6 pipeline stages) on a labeled test set and compares the
agent's final decision against ground truth. This measures the *agent's* accuracy,
not just the detector's — a wrong reasoning-stage call would show up here even if
perception was correct.

**To run this for real:** point `TEST_DIR` at the Roboflow `test/images` folder
(downloaded in `01_exploration.ipynb`) and build `GROUND_TRUTH` from the matching
YOLO label files (map `with_mask` → COMPLIANT, `without_mask`/`incorrectly_worn_mask` → NON_COMPLIANT).
Aim for at least 10-20 scenarios per the assignment requirement.

In [ ]:
TEST_DIR = "../data/sample"  # swap for the Roboflow test/images folder for a full 10-20 scenario run

# Ground truth for the 3 bundled sample images (manually confirmed)
GROUND_TRUTH = {
    "caregiver_car_nomask.jpg": "NON_COMPLIANT",
    "caregiver_home_nomask.jpg": "NON_COMPLIANT",
    "caregiver_clinic_masked.jpg": "COMPLIANT",
}

agent = PPEComplianceAgent(
    weights_path="../models/trained/best_yolov8n_ppe.pt",  # falls back gracefully if missing
    results_dir="../results",
)
traces = agent.run(TEST_DIR)

In [ ]:
import os

correct = 0
total = 0
rows = []
for t in traces:
    fname = os.path.basename(t["image_path"])
    truth = GROUND_TRUTH.get(fname)
    if truth is None:
        continue
    predicted = t["status"]
    is_correct = predicted == truth
    correct += int(is_correct)
    total += 1
    rows.append((fname, truth, predicted, is_correct))

print(f"{'Image':35s} {'Ground Truth':15s} {'Predicted':15s} {'Correct'}")
for r in rows:
    print(f"{r[0]:35s} {r[1]:15s} {r[2]:15s} {r[3]}")

print(f"\nTask success rate: {correct}/{total} = {correct/max(total,1)*100:.1f}%")
print("NOTE: this bundled run uses only 3 sample images as a smoke test.")
print("Re-run with TEST_DIR pointed at the full Roboflow test/ split for the required 10-20 scenario evaluation.")

## 3. Robustness — Bad Input Handling

In [ ]:
import os
from PIL import Image

os.makedirs("../data/robustness_test", exist_ok=True)

# corrupt file (garbage bytes with .jpg extension)
with open("../data/robustness_test/corrupt.jpg", "w") as f:
    f.write("this is not an image")

# tiny image, below usable size
Image.new("RGB", (10, 10), "white").save("../data/robustness_test/tiny.jpg")

# valid but blank image (no useful content)
Image.new("RGB", (640, 640), "white").save("../data/robustness_test/blank.jpg")

robustness_traces = agent.run("../data/robustness_test")

for t in robustness_traces:
    print(f"{os.path.basename(t['image_path']):20s} -> {t['status']:25s} ({t['preprocessing']['reason']})")

**Result:** the corrupt file and the undersized image are both caught at the preprocessing
stage and marked `SKIPPED_INVALID_INPUT` — the agent logs why and moves on to the next
image rather than crashing. The blank (but valid) image is processed normally and correctly
returns `NO_DETECTION`, since there's genuinely nothing to detect — the agent abstains
instead of guessing.

## 4. Efficiency

Average per-image latency is written to `results/metrics.txt` after every run
(`agent._write_batch_summary()`). Check that file for the running average across all
batches, or compute it directly from the traces below.

In [ ]:
latencies = [t["latency_sec"] for t in traces if "latency_sec" in t]
if latencies:
    print(f"Average latency: {sum(latencies)/len(latencies):.3f} sec/image")
    print(f"Min: {min(latencies):.3f}s | Max: {max(latencies):.3f}s")

## 5. Honest Failure Analysis

**Required: at least 2 documented failure cases with explanation.** Fill this in after
running the full 10-20 scenario evaluation above. Two known/anticipated failure modes
to look for:

1. **Occluded or side-profile faces** — the detector was trained primarily on
   front-facing photos. A caregiver photographed at a sharp angle, or with a face
   partially blocked (by a phone, hand, or camera position), may produce a
   `NO_DETECTION` result even though a person is clearly present in frame —
   the agent correctly abstains rather than guessing, but this reduces practical
   coverage in real deployment.

2. **Domain mismatch between training data and real home-visit conditions** —
   the training dataset is general-purpose mask-detection imagery, not photos
   specifically taken in home-care settings (varied indoor lighting, phone camera
   selfie angles, car interiors, etc.). Confidence scores are noticeably more
   variable on these “in-the-wild” style photos (see `results/images/`) than on
   the clean, front-facing validation images the model was scored on during training.

*(Add specific example image filenames + agent output here once the full evaluation
run above has been executed against the Roboflow test set.)*